# Task 3 — GenAI Summarization

## Goal

Generate concise summaries for the final review clusters discovered in Task 2.

Each summary should describe the main themes, common customer feedback, and notable sentiment patterns inside a cluster.

The summaries will be based on the clustered review text and the final human-readable cluster interpretation.

## Daily Recap

### Previous work

Task 2 — Clustering is complete.

We tested KMeans with K = 4, 5 and 6 using TF-IDF features from `reviews.text`.

K = 6 was selected because it achieved the highest Silhouette Score among the tested values and produced the clearest semantic separation.

Four cluster themes are already aligned between both team analyses:

- Kindle / Reading
- Ease of Use / Setup
- Kids / Family / Gifts
- Tablets / Value

Two broader cluster names are still being aligned between team members.

### Today

We begin Task 3 — GenAI Summarization.

The goal is to create one concise summary per final cluster based on real customer reviews.

CUSTOMER REVIEWS
        │
        ├──── sentiment model
        │        ↓
        │   positive / neutral / negative
        │
        ├──── clustering
        │        ↓
        │   recurring topics
        │
        └──── GenAI summarization
                 ↓
           business insight

In [16]:
# Install the libraries required to run T5 locally.
%pip install torch transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


# Task 3 — Generative Summarization

## 1. Load the clustered dataset

### Question

What data from Task 2 do we need for summarization?

### What we did

We loaded `clustered_reviews.csv`, which contains the cleaned reviews together with:

- sentiment labels
- numerical cluster IDs
- human-readable cluster names

### Result

The dataset contains 34,615 reviews and the six final clusters from Task 2.

### Decision / Finding

Task 3 will use the clusters discovered in Task 2 as the basis for generating summaries.

In [17]:
# Import pandas for working with the clustered review data.
import pandas as pd

# Import Path to build reliable file paths.
from pathlib import Path

# Find the project root folder.
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is running from the notebooks folder,
# move one level up to the project root.
if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Build the path to the clustered dataset from Task 2.
CLUSTERED_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "clustered_reviews.csv"
)

# Load the clustered reviews.
df = pd.read_csv(
    CLUSTERED_DATA_PATH,
    low_memory=False
)

# Check the dataset dimensions.
print("Dataset shape:", df.shape)

# Confirm that the clustering columns exist.
assert "cluster" in df.columns
assert "cluster_name" in df.columns

print("Clustered dataset loaded successfully.")

Dataset shape: (34615, 9)
Clustered dataset loaded successfully.


In [18]:
# Update the cluster names to the final
# semantic names agreed by the team.
cluster_names = {
    0: "Kindle & Reading",
    1: "Ease of Use & Setup",
    2: "Fire TV & Streaming",
    3: "General & Mixed Feedback",
    4: "Kids, Family & Gifts",
    5: "Tablets & Value"
}

# Replace the old readable names using the cluster ID.
df["cluster_name"] = df["cluster"].map(cluster_names)

# Check that the final names are now correct.
df[
    ["cluster", "cluster_name"]
].drop_duplicates().sort_values("cluster")

,cluster,cluster_name
6,0,Kindle & Reading
7,1,Ease of Use & Setup
0,2,Fire TV & Streaming
32,3,General & Mixed Feedback
1,4,"Kids, Family & Gifts"
2,5,Tablets & Value


## 1. Prepare representative review samples

### Question

How should we choose reviews for GenAI summarization?

### Decision

We use sentiment-aware sampling instead of purely random sampling.

The original dataset is strongly skewed toward Positive reviews. A random sample could therefore miss important Neutral and Negative feedback.

For each cluster, we aim to sample:

- 20 Positive reviews
- 6 Neutral reviews
- 4 Negative reviews

This keeps the sample mostly representative of the dataset while ensuring minority feedback is visible to the summarization model.

In [19]:
# Number of reviews we want from each sentiment group.
POSITIVE_SAMPLE = 20
NEUTRAL_SAMPLE = 6
NEGATIVE_SAMPLE = 4

# Create an empty dictionary.
# Each cluster ID will store a small representative sample.
cluster_samples = {}

# Loop through every cluster ID in sorted order.
for cluster_id in sorted(df["cluster"].unique()):

    # Select only reviews from the current cluster.
    cluster_df = df[
        df["cluster"] == cluster_id
    ]

    # Select Positive reviews from this cluster.
    positive_reviews = cluster_df[
        cluster_df["sentiment"] == "positive"
    ]

    # Select Neutral reviews from this cluster.
    neutral_reviews = cluster_df[
        cluster_df["sentiment"] == "neutral"
    ]

    # Select Negative reviews from this cluster.
    negative_reviews = cluster_df[
        cluster_df["sentiment"] == "negative"
    ]

    # Sample Positive reviews.
    positive_sample = positive_reviews.sample(
        n=min(POSITIVE_SAMPLE, len(positive_reviews)),
        random_state=42
    )

    # Sample Neutral reviews.
    neutral_sample = neutral_reviews.sample(
        n=min(NEUTRAL_SAMPLE, len(neutral_reviews)),
        random_state=42
    )

    # Sample Negative reviews.
    negative_sample = negative_reviews.sample(
        n=min(NEGATIVE_SAMPLE, len(negative_reviews)),
        random_state=42
    )

    # Combine the three sentiment groups.
    representative_sample = pd.concat(
        [
            positive_sample,
            neutral_sample,
            negative_sample
        ]
    )

    # Store only the review text in the dictionary.
    cluster_samples[cluster_id] = (
        representative_sample["reviews.text"]
        .tolist()
    )

# Check how many reviews were stored for each cluster.
for cluster_id, reviews in cluster_samples.items():
    print(
        f"Cluster {cluster_id}:",
        len(reviews),
        "reviews"
    )

Cluster 0: 30 reviews
Cluster 1: 30 reviews
Cluster 2: 30 reviews
Cluster 3: 30 reviews
Cluster 4: 30 reviews
Cluster 5: 30 reviews


## 2. Build the summarization prompt

### Question

How do we give the LLM enough context to summarize a cluster correctly?

### What we do

For each cluster, we provide:

- the cluster name
- a representative sample of reviews
- explicit instructions about what information to extract

The goal is to reduce vague outputs and avoid hallucinated details.

In [20]:
# Create a function that builds one prompt
# for one review cluster.
def build_cluster_prompt(
    cluster_name,
    reviews
):
    # Convert the list of reviews into one
    # numbered block of text.
    reviews_text = ""

    for index, review in enumerate(
        reviews,
        start=1
    ):
        reviews_text += (
            f"{index}. {review}\n"
        )

    # Build the final prompt.
    prompt = f"""
You are analyzing customer reviews.

Cluster:
{cluster_name}

Below is a representative sample of reviews from this cluster.

Reviews:
{reviews_text}

Your task:

1. Identify the main themes in the reviews.
2. Describe the most common positive feedback.
3. Describe the most common complaints or concerns.
4. Write a short business-oriented summary of what customers are saying.
5. Use only information supported by the reviews.
6. Do not invent details that are not present in the reviews.

Keep the response concise and clear.
"""

    return prompt

In [21]:
# Select one cluster ID for testing.
test_cluster_id = 0

# Find the readable name for that cluster.
test_cluster_name = (
    df[
        df["cluster"] == test_cluster_id
    ]["cluster_name"]
    .iloc[0]
)

# Build the prompt for this test cluster.
test_prompt = build_cluster_prompt(
    test_cluster_name,
    cluster_samples[test_cluster_id]
)

# Print only the first part of the prompt
# so we can inspect it before sending it to an LLM.
print(
    test_prompt[:3000]
)


You are analyzing customer reviews.

Cluster:
Kindle & Reading

Below is a representative sample of reviews from this cluster.

Reviews:
1. This e-reader is amazing. It is so light and easy to read with one hand. The screen is great and the battery lasts for weeks.
2. With a new higher resolution display, Bookerly font, and long lasting battery, the Kindle Paperwhite is the best e-readr in the market today. Reads like real paper. The device capabilities are complemented by Kindle's vast e-book store.
3. Perfect products with perfect recommendations. I enjoy with kindle. Thank you
4. I just love the size of the kindle and it's easy to take on the go. Perfect reading for outdoors on the deck around the pool.
5. This is great I was used to just the plain Kindle and this feels more like a tablet than just a kindle. I use it to check email, do pintrest and read my books. Totally in love with my new Kindle fire!
6. I've always been a big proponent of real books; love the smell, touch, etc, 

# 2. Create a balanced qualitative sample

### Question

How should we select reviews for testing the generative model?

### What we did

For each cluster, we selected:

- 5 Positive reviews
- 5 Neutral reviews
- 5 Negative reviews

This creates 15 reviews per cluster and 90 reviews in total.

### Result

Each of the six clusters contains an equal qualitative sample of the three sentiment groups.

### Decision / Finding

This balanced sample is used only to evaluate the quality of the generated summaries.

It must not be used to estimate the real sentiment proportions of a cluster. Real sentiment statistics come from the full dataset.

In [25]:
# Create a balanced qualitative sample:
# 5 positive, 5 neutral and 5 negative reviews
# from each cluster.

samples = []

for cluster_id in sorted(df["cluster"].unique()):

    # Select only reviews from the current cluster.
    cluster_df = df[
        df["cluster"] == cluster_id
    ]

    # Go through each sentiment category.
    for sentiment in [
        "positive",
        "neutral",
        "negative"
    ]:

        # Select only reviews with this sentiment.
        sentiment_df = cluster_df[
            cluster_df["sentiment"] == sentiment
        ]

        # Take up to 5 reviews.
        n = min(
            5,
            len(sentiment_df)
        )

        # Only sample if reviews exist.
        if n > 0:
            sample = sentiment_df.sample(
                n=n,
                random_state=42
            )

            samples.append(sample)

# Combine all sampled reviews into one DataFrame.
summary_sample = pd.concat(
    samples,
    ignore_index=True
)

# Check the result.
print(
    "Balanced qualitative sample shape:",
    summary_sample.shape
)

summary_sample[
    ["cluster", "cluster_name", "sentiment"]
].value_counts().sort_index()

Balanced qualitative sample shape: (90, 9)


cluster  cluster_name              sentiment
0        Kindle & Reading          negative     5
                                   neutral      5
                                   positive     5
1        Ease of Use & Setup       negative     5
                                   neutral      5
                                   positive     5
2        Fire TV & Streaming       negative     5
                                   neutral      5
                                   positive     5
3        General & Mixed Feedback  negative     5
                                   neutral      5
                                   positive     5
4        Kids, Family & Gifts      negative     5
                                   neutral      5
                                   positive     5
5        Tablets & Value           negative     5
                                   neutral      5
                                   positive     5
Name: count, dtype: int64

## 3. Prepare the local generative-model environment

### Question

Can we run a pretrained generative model locally without using an external API?

### What we did

We installed and verified:

- PyTorch
- Hugging Face Transformers
- SentencePiece

We also checked whether CUDA GPU acceleration is available.

### Result

PyTorch and Transformers are working correctly.

CUDA is not available, so the model will run on CPU.

### Decision / Finding

CPU is sufficient for this first baseline because we are only performing inference with a small pretrained model, not training or fine-tuning it.

## 3. Prepare the local generative-model environment

### Question

Can we run a pretrained generative model locally without using an external API?

### What we did

We installed and verified:

- PyTorch
- Hugging Face Transformers
- SentencePiece

We also checked whether CUDA GPU acceleration is available.

### Result

PyTorch and Transformers are working correctly.

CUDA is not available, so the model will run on CPU.

### Decision / Finding

CPU is sufficient for this first baseline because we are only performing inference with a small pretrained model, not training or fine-tuning it.

In [22]:
# Check which Python environment is running
# and whether the required libraries are available.
import sys
import torch
import transformers

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: /Users/caiomaia/ironhack-workspace/.venv/bin/python3.12.real
PyTorch: 2.14.1
Transformers: 5.18.0
CUDA available: False


## 4. Load the pretrained T5-small model

### Question

Which pretrained generative model will we use for the first summarization baseline?

### What we did

We loaded `google-t5/t5-small` using Hugging Face Transformers.

T5 is a sequence-to-sequence model: it receives text as input and generates text as output.

### Result

The tokenizer and pretrained T5-small model loaded successfully.

### Decision / Finding

T5-small will be our first local summarization baseline.

No fine-tuning is used at this stage.

In [23]:
# Import the tokenizer and the correct model class
# for T5, which is a sequence-to-sequence model.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Name of the pretrained model on Hugging Face.
model_name = "google-t5/t5-small"

# Load the tokenizer.
# The tokenizer converts text into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pretrained T5-small model.
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("T5 model loaded successfully!")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 12805.69it/s]

T5 model loaded successfully!


## 5. Validate basic T5 inference

### Question

Can T5-small successfully generate a summary before we use real customer reviews?

### What we did

We gave the model a short synthetic Kindle example containing both positive feedback and complaints.

### Result

T5 produced a valid summary and preserved both the positive and negative information without inventing new facts.

However, the output was close to the original wording and showed limited abstraction.

### Decision / Finding

The inference pipeline works correctly.

T5-small is usable as a baseline, but its summarization quality still needs to be evaluated on real review data.

In [24]:
# Create a very small test input first.
# T5 expects the instruction "summarize:" before the text.
test_text = """
summarize: Customers like the Kindle because it is lightweight,
easy to use and convenient for reading at night.
Some customers complain about advertisements,
limited app availability and occasional software problems.
"""

# Convert the text into tokens that T5 can process.
inputs = tokenizer(
    test_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Ask T5 to generate a summary.
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    min_new_tokens=15,
    num_beams=4,
    early_stopping=True
)

# Convert the generated tokens back into readable text.
summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(summary)

the Kindle is lightweight, easy to use and convenient for reading at night. some customers complain about advertisements, limited app availability and occasional software problems.


## 6. Test T5 on one real cluster

### Question

Can T5-small produce a useful summary from real reviews in the `Kindle & Reading` cluster?

### What we did

We selected the 15 balanced Kindle reviews:

- 5 Positive
- 5 Neutral
- 5 Negative

Before generating a summary, we inspected the full review text and measured the model input length.

In [26]:
# Select only reviews from the Kindle & Reading cluster.
kindle_df = summary_sample[
    summary_sample["cluster_name"] == "Kindle & Reading"
]

# Show the sentiment and review text
# so we know exactly what T5 will receive.
kindle_df[
    ["sentiment", "reviews.text"]
].reset_index(drop=True)

,sentiment,reviews.text
0,positive,This e-reader is amazing. It is so light and e...
1,positive,"With a new higher resolution display, Bookerly..."
2,positive,Perfect products with perfect recommendations....
3,positive,I just love the size of the kindle and it's ea...
4,positive,This is great I was used to just the plain Kin...
5,neutral,I had to replace my Kindle 3G keyboard after s...
6,neutral,Its a decent reader for beginners. There is no...
7,neutral,The Touch cover from Amazon has some good and ...
8,neutral,I got it to store all my college books without...
9,neutral,"Everyone knows ""kindle fire"". When these went ..."


In [27]:
# Print every Kindle review in full.
# This lets us inspect the exact evidence
# that will later be given to T5.

for index, row in kindle_df.reset_index(drop=True).iterrows():

    # Print the review number and sentiment.
    print(
        f"Review {index + 1} "
        f"({row['sentiment']}):"
    )

    # Print the complete review text.
    print(row["reviews.text"])

    # Add a separator to make the reviews easier to read.
    print("-" * 80)

Review 1 (positive):
This e-reader is amazing. It is so light and easy to read with one hand. The screen is great and the battery lasts for weeks.
--------------------------------------------------------------------------------
Review 2 (positive):
With a new higher resolution display, Bookerly font, and long lasting battery, the Kindle Paperwhite is the best e-readr in the market today. Reads like real paper. The device capabilities are complemented by Kindle's vast e-book store.
--------------------------------------------------------------------------------
Review 3 (positive):
Perfect products with perfect recommendations. I enjoy with kindle. Thank you
--------------------------------------------------------------------------------
Review 4 (positive):
I just love the size of the kindle and it's easy to take on the go. Perfect reading for outdoors on the deck around the pool.
--------------------------------------------------------------------------------
Review 5 (positive):
This

In [28]:
# Combine the 15 Kindle reviews into one text block.
kindle_reviews_text = "\n".join(
    kindle_df["reviews.text"].astype(str)
)

# Add the T5 summarization instruction.
kindle_input = (
    "summarize: "
    + kindle_reviews_text
)

# Tokenize without truncating so we can measure
# how large the complete input actually is.
kindle_tokens = tokenizer(
    kindle_input,
    truncation=False
)

# Count the number of input tokens.
token_count = len(
    kindle_tokens["input_ids"]
)

print("Kindle input token count:", token_count)
print("Current T5 input limit used in our code: 512")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1076 > 512). Running this sequence through the model will result in indexing errors


Kindle input token count: 1076
Current T5 input limit used in our code: 512


INPUT
All 15 real Kindle reviews.
TRANSFORMATION
The tokenizer converts the complete text into T5 tokens, but this time we do not cut anything.
OUTPUT
Something like:
Kindle input token count: 900
Current T5 input limit used in our code: 512

WHY
Before generating a summary, we need to know whether T5 can actually see all the evidence we selected.
This is an important AI-engineering lesson:
A balanced dataset sample is useless if the model's context window silently removes part of it.

Run that token-count cell and send me the number. Then we'll decide how to construct the T5 input without losing the balance.

In [29]:
# Measure the token size of each sentiment group
# inside the Kindle & Reading sample.

for sentiment in ["positive", "neutral", "negative"]:

    sentiment_text = "\n".join(
        kindle_df[
            kindle_df["sentiment"] == sentiment
        ]["reviews.text"].astype(str)
    )

    sentiment_input = (
        "summarize: "
        + sentiment_text
    )

    sentiment_tokens = tokenizer(
        sentiment_input,
        truncation=False
    )

    token_count = len(
        sentiment_tokens["input_ids"]
    )

    print(
        sentiment,
        "token count:",
        token_count
    )

positive token count: 195
neutral token count: 731
negative token count: 156


In [30]:
# Measure the token length of each neutral Kindle review.

neutral_reviews = kindle_df[
    kindle_df["sentiment"] == "neutral"
].reset_index(drop=True)

for index, review in neutral_reviews["reviews.text"].items():

    review_input = (
        "summarize: "
        + str(review)
    )

    review_tokens = tokenizer(
        review_input,
        truncation=False
    )

    token_count = len(
        review_tokens["input_ids"]
    )

    print(
        f"Neutral review {index + 1}:",
        token_count,
        "tokens"
    )

Neutral review 1: 174 tokens
Neutral review 2: 43 tokens
Neutral review 3: 361 tokens
Neutral review 4: 81 tokens
Neutral review 5: 84 tokens


In [31]:
# Combine the 5 positive Kindle reviews.
positive_text = "\n".join(
    kindle_df[
        kindle_df["sentiment"] == "positive"
    ]["reviews.text"].astype(str)
)

# Add the T5 task instruction.
positive_input = (
    "summarize: "
    + positive_text
)

# Convert the text into PyTorch tensors.
inputs = tokenizer(
    positive_input,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate the summary.
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    min_new_tokens=15,
    num_beams=4,
    early_stopping=True
)

# Convert generated tokens back into readable text.
positive_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(positive_summary)

the Kindle Paperwhite is the best e-readr in the market today. it is so light and easy to read with one hand. the screen is great and the battery lasts for weeks.


### Result

The complete Kindle sample contains 1,076 tokens.

The current T5-small input limit is 512 tokens.

The sentiment groups contain:

- Positive: 195 tokens
- Neutral: 731 tokens
- Negative: 156 tokens

The Neutral group exceeds the model input limit because some individual reviews are much longer than the others.

### Decision / Finding

We should not concatenate all 15 reviews and rely on truncation, because important evidence would be discarded.

Instead, we will test smaller chunks and evaluate their summaries before building a full multi-step summarization process.